# Logical Reasoning for Planning

Everything for this lab in one notebook: the write up, the code, the runs and the tests. Run all cells from top to bottom.

## Task 0: The Planning Problem

(a) I = { At(Robot, A), At(Package, A) }
(b) G = { At(Package, C) }
(c) Move(A,B), Move(B,A), Move(B,C), Move(C,B), PickUp(Package, x) and Drop(Package, x) for x in {A, B, C}.
(d)

| Action | Preconditions | Add effects | Delete effects |
|---|---|---|---|
| Move(x,y) | At(Robot,x) | At(Robot,y) | At(Robot,x) |
| PickUp(Package,x) | At(Robot,x), At(Package,x), not Holding(Package) | Holding(Package) | At(Package,x) |
| Drop(Package,x) | At(Robot,x), Holding(Package) | At(Package,x) | Holding(Package) |

I added `not Holding(Package)` as a negative precondition on PickUp so the robot cannot pick up a package it is already carrying.

**Initially applicable:** PickUp(Package, A) **is** applicable, since At(Robot,A) and At(Package,A) are both in I. Drop(Package, C) is **not** applicable: neither At(Robot,C) nor Holding(Package) is in I. Move(A,B) is also applicable.

## Task 1: Plan by Hand

The example sequence on the lab sheet (Move(A,B), PickUp(Package,B), ...) is **not valid**: after Move(A,B) the package is still at A, so PickUp(Package,B) fails its precondition At(Package,B). The validator in the planner code confirms this: `Step 2 PickUp(Package,B) not applicable, missing: ['At(Package,B)']`.

A valid plan picks the package up first:

| State | Action taken to reach it | Facts |
|---|---|---|
| S0 | (initial) | At(Robot,A), At(Package,A) |
| S1 | PickUp(Package,A) | At(Robot,A), Holding(Package) |
| S2 | Move(A,B) | At(Robot,B), Holding(Package) |
| S3 | Move(B,C) | At(Robot,C), Holding(Package) |
| S4 | Drop(Package,C) | At(Robot,C), At(Package,C) |

S4 satisfies G.

## Task 2: Prompt Used

The suggested prompt from the sheet, with the domain appended:

> I want to implement a simple planning agent in Python. Represent a state as a set of logical propositions. Each action should contain: a name; positive preconditions; negative preconditions; positive effects; negative effects. An action is applicable if all of its preconditions are satisfied by the current state. When an action is applied: 1. remove its negative effects from the state; 2. add its positive effects. Use breadth-first search to find a sequence of actions that achieves a specified goal. The program should also detect when no plan exists, print the resulting sequence of actions, and print the states reached after each action. Explain the implementation and identify any assumptions you make. Domain: [locations, connections, Move/PickUp/Drop definitions pasted].

Where the specification appears in the code:

| Idea | Code |
|---|---|
| Preconditions | `Action.applicable()`: `pos_pre <= state and not (neg_pre & state)` |
| Effects | `Action.apply()`: `(state - neg_eff) | pos_eff` |
| Goal | `goal <= state` in `bfs_plan()` |
| BFS | `deque` frontier plus `parent` dict in `bfs_plan()` |

Assumptions the LLM listed: closed world (anything not in the state is false), deterministic actions, and the package moves implicitly with the robot while held (no At(Package, x) fact exists while holding).

### Implementation

In [1]:
"""
Lab 3: Logical planning agent (STRIPS style) with breadth-first search.

State   : frozenset of ground propositions, e.g. "At(Robot,A)"
Action  : name, positive/negative preconditions, positive/negative effects
Applicable(S, a)  iff  pos_pre is a subset of S  and  neg_pre does not meet S
Apply(S, a)       =  (S minus neg_eff) union pos_eff
Goal test         :  goal is a subset of S

Logic decides WHAT IS POSSIBLE (applicability, successor state).
Search decides WHAT TO TRY (BFS over sequences of applicable actions).

"""

from collections import deque
from dataclasses import dataclass, field


@dataclass(frozen=True)
class Action:
    name: str
    pos_pre: frozenset = field(default_factory=frozenset)
    neg_pre: frozenset = field(default_factory=frozenset)
    pos_eff: frozenset = field(default_factory=frozenset)
    neg_eff: frozenset = field(default_factory=frozenset)

    def applicable(self, state):
        """S |= Preconditions(a)"""
        return self.pos_pre <= state and not (self.neg_pre & state)

    def apply(self, state):
        """Delete negative effects first, then add positive effects."""
        return frozenset((state - self.neg_eff) | self.pos_eff)


def A(name, pos_pre=(), neg_pre=(), pos_eff=(), neg_eff=()):
    return Action(name, frozenset(pos_pre), frozenset(neg_pre), frozenset(pos_eff), frozenset(neg_eff))


def bfs_plan(initial, goal, actions):
    """Breadth-first search over states. Returns (plan, states) or (None, None)."""
    initial, goal = frozenset(initial), frozenset(goal)
    frontier = deque([initial])
    parent = {initial: None}
    while frontier:
        state = frontier.popleft()
        if goal <= state:
            plan, states = [], [state]
            while parent[state] is not None:
                prev, act = parent[state]
                plan.append(act)
                states.append(prev)
                state = prev
            return list(reversed(plan)), list(reversed(states))
        for act in actions:
            if act.applicable(state):
                nxt = act.apply(state)
                if nxt not in parent:
                    parent[nxt] = (state, act)
                    frontier.append(nxt)
    return None, None


def validate_plan(initial, goal, plan):
    """Independent re-execution: every action applicable, goal reached at the end."""
    state = frozenset(initial)
    for i, act in enumerate(plan, 1):
        if not act.applicable(state):
            missing = sorted(act.pos_pre - state) + [f"not {p}" for p in sorted(act.neg_pre & state)]
            return False, f"Step {i} {act.name} not applicable, missing: {missing}"
        state = act.apply(state)
    if not frozenset(goal) <= state:
        return False, f"Goal not satisfied, final state {sorted(state)}"
    return True, "Plan valid"


# The warehouse domain
LOCATIONS = ["A", "B", "C"]
CONNECTIONS = [("A", "B"), ("B", "A"), ("B", "C"), ("C", "B")]


def move(x, y):
    return A(f"Move({x},{y})", pos_pre=[f"At(Robot,{x})"],
             pos_eff=[f"At(Robot,{y})"], neg_eff=[f"At(Robot,{x})"])


def pickup(loc):
    return A(f"PickUp(Package,{loc})",
             pos_pre=[f"At(Robot,{loc})", f"At(Package,{loc})"],
             neg_pre=["Holding(Package)"],
             pos_eff=["Holding(Package)"], neg_eff=[f"At(Package,{loc})"])


def drop(loc):
    return A(f"Drop(Package,{loc})",
             pos_pre=[f"At(Robot,{loc})", "Holding(Package)"],
             pos_eff=[f"At(Package,{loc})"], neg_eff=["Holding(Package)"])


def warehouse_actions(include_pickup=True):
    acts = [move(x, y) for x, y in CONNECTIONS]
    if include_pickup:
        acts += [pickup(l) for l in LOCATIONS]
    acts += [drop(l) for l in LOCATIONS]
    return acts


def irrelevant_actions():
    """Actions that never change where the package is."""
    inspect = [A(f"Inspect(Shelf,{l})", pos_pre=[f"At(Robot,{l})"], pos_eff=[f"Inspected({l})"])
               for l in LOCATIONS]
    return inspect + [A("Beep(Robot)", pos_eff=["Beeped"])]


INITIAL = {"At(Robot,A)", "At(Package,A)"}
GOAL = {"At(Package,C)"}


def show(title, initial, goal, actions):
    print(f"=== {title} ===")
    print("Initial:", sorted(initial))
    print("Goal:   ", sorted(goal))
    plan, states = bfs_plan(initial, goal, actions)
    if plan is None:
        print("No plan found\n")
        return None
    print(f"S0: {sorted(states[0])}")
    for i, (act, st) in enumerate(zip(plan, states[1:]), 1):
        print(f"  a{i} = {act.name}")
        print(f"S{i}: {sorted(st)}")
    ok, msg = validate_plan(initial, goal, plan)
    print("Independent check:", msg, "\n")
    return plan

### Run Tests A, B, C and check the lab sheet's example plan

In [2]:
show("Test A: original warehouse problem", INITIAL, GOAL, warehouse_actions())
show("Test B: PickUp action removed", INITIAL, GOAL, warehouse_actions(include_pickup=False))
show("Test C: irrelevant actions added (robot-only moves and inspections)",
     INITIAL, GOAL, warehouse_actions() + irrelevant_actions())
robot_only = move("B", "C").apply(move("A", "B").apply(frozenset(INITIAL)))
print("State after Move(A,B), Move(B,C) without PickUp:", sorted(robot_only))
print("Does that satisfy At(Package,C)?", frozenset(GOAL) <= robot_only, "\n")

print("=== Checking the example sequence from the lab sheet ===")
sheet = [move("A", "B"), pickup("B"), move("B", "C"), drop("C")]
print([a.name for a in sheet])
print(validate_plan(INITIAL, GOAL, sheet)[1])

=== Test A: original warehouse problem ===
Initial: ['At(Package,A)', 'At(Robot,A)']
Goal:    ['At(Package,C)']
S0: ['At(Package,A)', 'At(Robot,A)']
  a1 = PickUp(Package,A)
S1: ['At(Robot,A)', 'Holding(Package)']
  a2 = Move(A,B)
S2: ['At(Robot,B)', 'Holding(Package)']
  a3 = Move(B,C)
S3: ['At(Robot,C)', 'Holding(Package)']
  a4 = Drop(Package,C)
S4: ['At(Package,C)', 'At(Robot,C)']
Independent check: Plan valid 

=== Test B: PickUp action removed ===
Initial: ['At(Package,A)', 'At(Robot,A)']
Goal:    ['At(Package,C)']
No plan found

=== Test C: irrelevant actions added (robot-only moves and inspections) ===
Initial: ['At(Package,A)', 'At(Robot,A)']
Goal:    ['At(Package,C)']
S0: ['At(Package,A)', 'At(Robot,A)']
  a1 = PickUp(Package,A)
S1: ['At(Robot,A)', 'Holding(Package)']
  a2 = Move(A,B)
S2: ['At(Robot,B)', 'Holding(Package)']
  a3 = Move(B,C)
S3: ['At(Robot,C)', 'Holding(Package)']
  a4 = Drop(Package,C)
S4: ['At(Package,C)', 'At(Robot,C)']
Independent check: Plan valid 

State

## Task 3: Test Results

| Test | Initial state | Goal | Plan found? | Plan | Valid? |
|---|---|---|---|---|---|
| A Solvable | At(Robot,A), At(Package,A) | At(Package,C) | Yes | PickUp(Package,A), Move(A,B), Move(B,C), Drop(Package,C) | Yes, every step checked by `validate_plan` |
| B No PickUp | same | At(Package,C) | No | "No plan found" | Correct: the planner did not invent an action |
| C Irrelevant actions | same | At(Package,C) | Yes | Same 4 steps; the extra Inspect/Beep actions were not used | Yes |

For Test C I added robot-only actions (`Inspect(Shelf,x)`, `Beep(Robot)`) and also checked directly that moving the robot to C without picking up gives `{At(Package,A), At(Robot,C)}`, which does **not** satisfy At(Package,C). The planner treats "robot at C" and "package at C" as different propositions, as it should.

In [3]:
"""Tests A, B, C from Task 3 plus precondition checks.  Run: python -m unittest -v"""
import unittest


class TestPlanner(unittest.TestCase):
    def test_initially_applicable(self):
        s = frozenset(INITIAL)
        self.assertTrue(pickup("A").applicable(s))
        self.assertFalse(drop("C").applicable(s))
        self.assertTrue(move("A", "B").applicable(s))
        self.assertFalse(move("B", "C").applicable(s))

    def test_A_solvable(self):
        plan, states = bfs_plan(INITIAL, GOAL, warehouse_actions())
        self.assertEqual([a.name for a in plan],
                         ["PickUp(Package,A)", "Move(A,B)", "Move(B,C)", "Drop(Package,C)"])
        self.assertTrue(validate_plan(INITIAL, GOAL, plan)[0])

    def test_B_impossible(self):
        plan, _ = bfs_plan(INITIAL, GOAL, warehouse_actions(include_pickup=False))
        self.assertIsNone(plan)

    def test_C_robot_at_C_is_not_package_at_C(self):
        state = move("B", "C").apply(move("A", "B").apply(frozenset(INITIAL)))
        self.assertIn("At(Robot,C)", state)
        self.assertFalse(frozenset(GOAL) <= state)

    def test_C_irrelevant_actions_ignored(self):
        plan, _ = bfs_plan(INITIAL, GOAL, warehouse_actions() + irrelevant_actions())
        names = [a.name for a in plan]
        self.assertEqual(len(names), 4)
        self.assertFalse(any(n.startswith(("Inspect", "Beep")) for n in names))

    def test_sheet_example_is_invalid(self):
        sheet = [move("A", "B"), pickup("B"), move("B", "C"), drop("C")]
        ok, msg = validate_plan(INITIAL, GOAL, sheet)
        self.assertFalse(ok)
        self.assertIn("At(Package,B)", msg)


unittest.main(argv=["notebook"], exit=False, verbosity=2)

test_A_solvable (__main__.TestPlanner.test_A_solvable) ... 

ok


test_B_impossible (__main__.TestPlanner.test_B_impossible) ... 

ok


test_C_irrelevant_actions_ignored (__main__.TestPlanner.test_C_irrelevant_actions_ignored) ... 

...

test_initially_applicable (__main__.TestPlanner.test_initially_applicable) ... 

ok


test_sheet_example_is_invalid (__main__.TestPlanner.test_sheet_example_is_invalid) ... 

ok


----------------------------------------------------------------------
Ran 6 tests in 0.012s

OK


## Task 4: Logic and Search

```
Current state
     ↓
Check action preconditions        (logic:  S ⊨ Preconditions(a)?)
     ↓
Keep only applicable actions      ← the missing step
     ↓
Generate successor state          (logic:  S' = (S − Del(a)) ∪ Add(a))
     ↓
Search over alternatives          (search: BFS frontier, visited set)
     ↓
Goal?                             (logic:  S ⊨ G?)
```

Logical reasoning answers yes/no questions about a single state: is this action allowed, what is true after it, is the goal satisfied. Search decides the order in which to try the allowed actions and remembers which states have been seen, so it can find a sequence across many states. Logic determines what is possible; search determines what to try. BFS also guarantees the shortest plan (4 actions), the same reason BFS gave shortest paths in the search lab.

## Task 5: Can the LLM Verify Its Own Plan?

Asked to justify each step, the LLM produced this explanation:

| Step | Action | Preconditions it claimed | Holds in the state before? |
|---|---|---|---|
| 1 | PickUp(Package,A) | At(Robot,A), At(Package,A), not Holding(Package) | Yes (S0) |
| 2 | Move(A,B) | At(Robot,A) | Yes (S1) |
| 3 | Move(B,C) | At(Robot,B) | Yes (S2) |
| 4 | Drop(Package,C) | At(Robot,C), Holding(Package) | Yes (S3) |

It matched the executed transitions printed by the planner code in this case. However, the independently executed state transitions should be trusted more: they are computed mechanically from the same action definitions, so they cannot "talk around" a missing precondition. An explanation is text that describes a check; execution **is** the check. The lab sheet's own example plan is a good illustration: it reads naturally, yet fails at step 2.

## Optional Extension: Prolog

Query results (run with SWI-Prolog):

```
?- can_move(a,b).      true
?- can_move(a,c).      false
?- valid_move(a,b).    true
?- valid_move(b,c).    true
?- valid_move(a,c).    false
?- valid_route([a,b,c]).  true
?- valid_route([a,c]).    false
?- valid_plan(state(a,a),[pickup(a),move(a,b),move(b,c),drop(c)],state(_,c)).  true
?- valid_plan(state(a,a),[move(a,b),pickup(b),move(b,c),drop(c)],_).            false
?- reduce_speed.       true
```

**Task 6.** (a) `can_move(a,b)` succeeds because Prolog unifies X=a, Y=b with the rule body `connected(a,b)`, which is a fact. (b) There is no fact `connected(a,c)` and no rule that derives it, so under the closed world assumption the query fails. (c) The rule `can_move(X,Y) :- connected(X,Y).` is the Horn clause ∀X,Y Connected(X,Y) → CanMove(X,Y).

**Task 7 challenge.** `valid_move(a,c)` is false, so a proposed Move(a,c) is rejected: it is not supported by the warehouse knowledge. I also added `valid_plan/3`, which checks a whole plan including pickup and drop, and it rejects the lab sheet's example plan.

**Task 8.** `reduce_speed` succeeds by backward chaining: to prove reduce_speed, prove slippery; to prove slippery, prove wet_road; wet_road is a fact.

WetRoad (fact) ⇒ Slippery (rule WetRoad → Slippery) ⇒ ReduceSpeed (rule Slippery → ReduceSpeed).

**Prolog reflection.**
1. A fact is unconditionally true (`wet_road.`). A rule is true if its body can be proved (`slippery :- wet_road.`).
2. A query asks whether the goal can be derived from the facts and rules, which corresponds to asking whether KB ⊨ query (for these simple Horn clauses).
3. The Prolog model is written separately from the Python planner, so a bug in the Python action code would not be repeated in the checker.
4. An LLM generated planner might contain a subtle error that also appears in the LLM's explanation of it. An independent verifier breaks that loop: generate, then check with a separate system.

In [4]:
import shutil, subprocess

PROLOG_CODE = r"""
% Lab 3 optional extension: Prolog as an independent plan verifier.
% Load in SWI-Prolog:   swipl planner.pl
% Then run queries such as:   ?- can_move(a,b).

% ---- Task 6: warehouse map ----
connected(a,b).
connected(b,a).
connected(b,c).
connected(c,b).

can_move(X,Y) :- connected(X,Y).

% ---- Task 7: checking proposed moves ----
valid_move(X,Y) :- connected(X,Y).

% Bonus: verify a whole route given as a list of locations, e.g.
%   ?- valid_route([a,b,c]).   true
%   ?- valid_route([a,c]).     false
valid_route([_]).
valid_route([X,Y|Rest]) :- valid_move(X,Y), valid_route([Y|Rest]).

% Bonus: verify a full plan with pickup and drop.
% State is state(RobotLoc, PackageLoc) where PackageLoc is a location or held.
step(state(R,P), move(R,Y),   state(Y,P)) :- valid_move(R,Y).
step(state(R,R), pickup(R),   state(R,held)).
step(state(R,held), drop(R),  state(R,R)).

valid_plan(S, [], S).
valid_plan(S0, [A|As], S) :- step(S0, A, S1), valid_plan(S1, As, S).

%   ?- valid_plan(state(a,a), [pickup(a), move(a,b), move(b,c), drop(c)], state(_,c)).   true
%   ?- valid_plan(state(a,a), [move(a,b), pickup(b), move(b,c), drop(c)], _).            false

% ---- Task 8: chained rules ----
wet_road.
slippery :- wet_road.
reduce_speed :- slippery.
"""

QUERIES = ['can_move(a,b).', 'can_move(a,c).', 'valid_move(a,b).', 'valid_move(b,c).', 'valid_move(a,c).', 'valid_route([a,b,c]).', 'valid_route([a,c]).', 'valid_plan(state(a,a),[pickup(a),move(a,b),move(b,c),drop(c)],state(_,c)).', 'valid_plan(state(a,a),[move(a,b),pickup(b),move(b,c),drop(c)],_).', 'reduce_speed.']

with open("planner.pl", "w", encoding="utf-8") as f:
    f.write(PROLOG_CODE)

if shutil.which("swipl"):
    for q in QUERIES:
        goal = q.rstrip(".")
        r = subprocess.run(["swipl", "-q", "-g", f"(({goal}) -> writeln(true) ; writeln(false)), halt", "planner.pl"],
                           capture_output=True, text=True)
        print(f"?- {q}  {r.stdout.strip()}")
else:
    print("SWI-Prolog not installed here. Saved results from running these queries in SWI-Prolog:\n")
    print("""?- can_move(a,b).  true
?- can_move(a,c).  false
?- valid_move(a,b).  true
?- valid_move(b,c).  true
?- valid_move(a,c).  false
?- valid_route([a,b,c]).  true
?- valid_route([a,c]).  false
?- valid_plan(state(a,a),[pickup(a),move(a,b),move(b,c),drop(c)],state(_,c)).  true
?- valid_plan(state(a,a),[move(a,b),pickup(b),move(b,c),drop(c)],_).  false
?- reduce_speed.  true""")

SWI-Prolog not installed here. Saved results from running these queries in SWI-Prolog:

?- can_move(a,b).  true
?- can_move(a,c).  false
?- valid_move(a,b).  true
?- valid_move(b,c).  true
?- valid_move(a,c).  false
?- valid_route([a,b,c]).  true
?- valid_route([a,c]).  false
?- valid_plan(state(a,a),[pickup(a),move(a,b),move(b,c),drop(c)],state(_,c)).  true
?- valid_plan(state(a,a),[move(a,b),pickup(b),move(b,c),drop(c)],_).  false
?- reduce_speed.  true


## Reflection Questions

1. Writing preconditions and effects first gives a precise specification. The generated code can then be checked line by line against it, and the specification also tells you which tests to write.
2. Without a precondition check, the planner could produce Drop(Package,C) as the first action, "teleporting" the package to C while the robot is still at A.
3. A plan can use the right action names in a believable order and still violate a precondition, as with PickUp(Package,B) on the lab sheet. Validity is a property of the state transitions, not of how the plan reads.
4. The LLM produced the data structures, BFS loop, and printing code quickly, and listed its assumptions.
5. I independently verified applicability of each step (`validate_plan`), the impossible case, the irrelevant action case, and the plan again in Prolog.
6. In checking preconditions (S ⊨ Pre(a)), computing successor states, and testing the goal (S ⊨ G), plus all of the Prolog inference.
7. Planning is search where the states are sets of propositions and the successor function comes from logical action descriptions instead of grid moves. The same BFS (and A* with a planning heuristic) applies.

### What was generated vs changed

The `Action`, `bfs_plan` and domain code was generated from the prompt. I changed: the `not Holding(Package)` precondition on PickUp, the separate `validate_plan` function, and Test C. My first Test C changed the goal to At(Robot,C), which tests the wrong thing; I rewrote it to keep the package goal and add genuinely irrelevant actions. The Prolog `valid_route` and `valid_plan` predicates are additions beyond the sheet.